# N3-Based DCParser Implementation 

Now let's implement the full DCParser using N3 rules instead of SPARQL queries.

In [66]:
import subprocess
import tempfile
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

@dataclass
class N3PlannerConfig:
    """Configuration for N3-based planner"""
    n3_reasoner: str = "eye"  # or "cwm"
    enable_proof_generation: bool = False
    reasoner_timeout: int = 30
    rules_directory: str = "rules_n3_unified"  # Unified rule chaining directory
    eye_path: str = "eye"  # Path to EYE reasoner executable
    timeout: int = 30  # Reasoner timeout in seconds
    n3_query_all: str = """
@prefix : <http://www.w3.org/2000/10/swap/log#> .
{ ?s ?p ?o } => { ?s ?p ?o } .
"""  # N3 query to extract all inferred triples
    
class N3PolicyChecker:
    """Represents a PolicyChecker using N3 reasoning"""
    
    def __init__(self, policy_uri: URIRef, dp_uri: URIRef):
        self.policy = policy_uri
        self.dp = dp_uri
        self.graph = Graph()
        
        # Bind namespaces
        self.graph.bind("tb", tb)
        self.graph.bind("tbox", tbox)
        self.graph.bind("abox", abox)
        self.graph.bind("odrl", odrl)
        
    def get_graph(self):
        return self.graph
        
print("✓ N3PlannerConfig and N3PolicyChecker defined")

✓ N3PlannerConfig and N3PolicyChecker defined


In [67]:
class N3DCParser:
    """
    Parse Data Contract Policies using N3 logic rules
    
    This replaces the SPARQL-based DCParser with declarative N3 reasoning
    """
    
    def __init__(self, dp: str, graph: Graph, config: Optional[N3PlannerConfig] = None):
        self.dp = dp
        self.g = graph
        self.config = config or N3PlannerConfig()
        self.attr_mappings = {}
        
        # Bind namespaces
        self.g.bind("tb", tb)
        self.g.bind("tbox", tbox)
        self.g.bind("abox", abox)
        self.g.bind("odrl", odrl)
        self.g.bind("dqv", dqv)
        self.g.bind("log", log)
        self.g.bind("math", math)
        self.g.bind("string", string)
        
    def _read_contracts(self):
        """
        Get the policies and mappings associated with a data product
        :return: tuple of (policies_list, mappings_dict)
        """
        dp_uri = abox[self.dp]
        contracts = self.g.objects(subject=dp_uri, predicate=tb.hasDC)
        policies_list = []
        mappings_dict = {}
        
        for contract in contracts:
            # Handle policies
            policies = self.g.objects(subject=contract, predicate=tb.hasPolicy)
            for policy in policies:
                policies_list.append(policy)
                
            # Handle mappings
            mappings = self.g.objects(subject=contract, predicate=tb.hasMapping)
            for mapping in mappings:
                mfrom = self.g.value(subject=mapping, predicate=tb.mfrom)
                mto = self.g.value(subject=mapping, predicate=tb.mto)
                if mfrom and mto:
                    mappings_dict[str(mto)] = str(mfrom)
                    
        self.attr_mappings = mappings_dict
        return policies_list, mappings_dict
    
    def _execute_n3_rules(self, subdirectory: str = "") -> Graph:
        """
        Execute N3 rules from the configured rules directory
        
        :param subdirectory: Optional subdirectory within rules_directory (empty for unified rules)
        :return: Inferred triples as Graph
        """
        rules_path = Path(self.config.rules_directory)
        if subdirectory:
            rules_path = rules_path / subdirectory
        
        if not rules_path.exists():
            print(f"⚠️ Rules directory not found: {rules_path}")
            return Graph()
        
        # Prepare data as temporary N3 file
        # Create a new graph and copy triples (Graph doesn't have .copy() method)
        data_graph = Graph()
        for s, p, o in self.g:
            data_graph.add((s, p, o))
        
        # Bind namespaces to data graph
        data_graph.bind("tb", tb)
        data_graph.bind("tbox", tbox)
        data_graph.bind("abox", abox)
        data_graph.bind("odrl", odrl)
        data_graph.bind("dqv", dqv)
        
        # Collect all .n3 rule files
        rule_files = list(rules_path.glob("*.n3"))
        
        if not rule_files:
            print(f"⚠️ No .n3 files found in {rules_path}")
            return Graph()
        
        # Write data to temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.n3', delete=False) as f:
            data_file = f.name
            f.write(data_graph.serialize(format='n3'))
        
        # Write query file (construct all triples)
        with tempfile.NamedTemporaryFile(mode='w', suffix='.n3', delete=False) as qf:
            query_file = qf.name
            qf.write(self.config.n3_query_all)
        
        try:
            # Build EYE command with all rule files
            cmd = [
                self.config.eye_path,
                data_file,
                *[str(rf) for rf in rule_files],
                "--query", query_file,
                "--nope"
            ]
            
            # Run EYE reasoner
            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=self.config.timeout
            )
            
            if result.returncode != 0:
                print(f"⚠️ EYE reasoner error: {result.stderr}")
                return Graph()
            
            # Parse output N3
            output_g = Graph()
            output_g.parse(data=result.stdout, format='n3')
            
            # Apply custom skolemization to replace RDFLib's blank nodes with clean URIs
            skolem_g = Graph()
            bnode_map = {}
            
            for s, p, o in output_g:
                # Skolemize subject
                if isinstance(s, BNode):
                    if s not in bnode_map:
                        # Create clean hash-based URI using abox namespace
                        hash_val = hashlib.md5(str(s).encode()).hexdigest()[:12]
                        bnode_map[s] = abox[f"genid-{hash_val}"]
                    s = bnode_map[s]
                
                # Skolemize object
                if isinstance(o, BNode):
                    if o not in bnode_map:
                        hash_val = hashlib.md5(str(o).encode()).hexdigest()[:12]
                        bnode_map[o] = abox[f"genid-{hash_val}"]
                    o = bnode_map[o]
                
                skolem_g.add((s, p, o))
            
            return skolem_g
            
        except subprocess.TimeoutExpired:
            print(f"⚠️ EYE reasoner timeout after {self.config.timeout}s")
            return Graph()
        except Exception as e:
            print(f"⚠️ Error running EYE reasoner: {e}")
            return Graph()
        finally:
            # Clean up temp files
            Path(data_file).unlink(missing_ok=True)
            Path(query_file).unlink(missing_ok=True)
    
    def _apply_attribute_mappings(self):
        """
        Apply attribute mappings to operations that reference CDM attributes
        """
        for cdm_attr, actual_attr in self.attr_mappings.items():
            # Find all triples with CDM attribute as object
            triples_to_update = []
            for s, p, o in self.g:
                if str(o) == cdm_attr:
                    triples_to_update.append((s, p, o))
            
            # Update triples
            for s, p, o in triples_to_update:
                self.g.remove((s, p, o))
                self.g.add((s, p, URIRef(actual_attr)))
    
    def parse_contracts(self) -> Graph:
        """
        Main parsing method - executes all N3 rules in a single pass.
        Rule chaining happens automatically via pattern matching.
        
        Returns:
            Graph containing PolicyCheckers and Operation chains
        """
        print(f"\n{'='*70}")
        print(f"N3-Based Policy Parsing for Data Product: {self.dp}")
        print(f"{'='*70}\n")
        
        # Step 1: Read contracts
        policies, mappings = self._read_contracts()
        print(f"✓ Found {len(policies)} policies")
        print(f"✓ Found {len(mappings)} attribute mappings")
        
        if mappings:
            print("\nAttribute Mappings:")
            for cdm_attr, actual_attr in mappings.items():
                cdm_name = cdm_attr.split('#')[-1].split('/')[-1]
                actual_name = actual_attr.split('#')[-1].split('/')[-1]
                print(f"  {cdm_name} → {actual_name}")
        
        # Step 2: Execute all unified rules (chaining happens automatically)
        print("\n" + "-"*70)
        print("Executing N3 rules with automatic chaining...")
        print("-"*70)
        
        result = self._execute_n3_rules("")  # Empty string = no subdirectory
        
        if len(result) > 0:
            self.g += result
            print(f"✓ Rules complete: {len(result)} triples inferred")
            
            # Count PolicyCheckers created
            pcs = list(self.g.subjects(RDF.type, tbox.PolicyChecker))
            print(f"✓ Created {len(pcs)} PolicyChecker(s)")
            
            # Count operations
            ops = list(self.g.subjects(RDF.type, tbox.Operation))
            print(f"✓ Total operations: {len(ops)}")
        else:
            print("⚠️ Rules produced no results")
        
        # Step 3: Apply attribute mappings
        if self.attr_mappings:
            print("\n" + "-"*70)
            print("Applying attribute mappings to operations...")
            print("-"*70)
            self._apply_attribute_mappings()
            print("✓ Attribute mappings applied")
        
        # Step 4: Summary
        print("\n" + "="*70)
        print("Parsing Complete!")
        print("="*70)
        print(f"Total triples in graph: {len(self.g)}")
        
        # Show PolicyChecker details
        pcs = list(self.g.subjects(RDF.type, tbox.PolicyChecker))
        if pcs:
            print(f"\nPolicyCheckers created:")
            for pc in pcs:
                print(f"\n  {pc}")
                for p, o in self.g.predicate_objects(pc):
                    pred_name = str(p).split('#')[-1].split('/')[-1]
                    obj_str = str(o).split('#')[-1].split('/')[-1] if isinstance(o, URIRef) else str(o)
                    print(f"    {pred_name}: {obj_str}")
                    
        return self.g

print("✓ N3DCParser class defined with custom abox: skolemization")

✓ N3DCParser class defined with custom abox: skolemization


# Testing with SDM 


In [70]:
# Load real SDM and test with UPENN data product (has 5 policies)
dp_name = "UPENN-GBM_clinical_info_v21csv"

sdm_path = Path("../../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl")

if sdm_path.exists():
    real_sdm = Graph()
    real_sdm.parse(str(sdm_path), format='turtle')
    
    print(f"✓ Loaded real SDM: {len(real_sdm)} triples")
    print(f"✓ Using data product: {dp_name}")
    
    if reasoner_available:
        # Create parser
        real_parser = N3DCParser(dp_name, real_sdm, config)
        
        # Parse
        result = real_parser.parse_contracts()
        
        # Save result
        output_path = Path("../../../FederatedComputationalGovernance/SemanticDataModel/sdm_with_pcs.ttl")
        result.serialize(destination=str(output_path), format='turtle')
        print(f"\n✓ Saved enhanced SDM to: {output_path}")
        
    else:
        print("\n⚠️ Skipping (reasoner not available)")
else:
    print(f"⚠️ SDM not found at {sdm_path}")


✓ Loaded real SDM: 474 triples
✓ Using data product: UPENN-GBM_clinical_info_v21csv

N3-Based Policy Parsing for Data Product: UPENN-GBM_clinical_info_v21csv

✓ Found 0 policies
✓ Found 0 attribute mappings

----------------------------------------------------------------------
Executing N3 rules with automatic chaining...
----------------------------------------------------------------------
✓ Rules complete: 629 triples inferred
✓ Created 7 PolicyChecker(s)
✓ Total operations: 16

Parsing Complete!
Total triples in graph: 630

PolicyCheckers created:

  http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#genid-7d16ad2c9a6a
    type: PolicyChecker
    validates: UPENN-GBM_clinical_info_v21csv
    accordingTo: p1
    nextStep: genid-a20c107ab178

  http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#genid-e009edcdbf7e
    type: PolicyChecker
    accordingTo: p1
    validates: 0002DCM
    nextStep: genid-14bcb5cf1ba8

  http://www.semanticweb.org/acraf/ontologi

## 13. Validate PolicyCheckers with SHACL <a id="validate-pcs"></a>

Test that generated PolicyCheckers conform to expected structure.

In [ ]:
# Validate PolicyCheckers with SHACL
try:
    from pyshacl import validate
    
    print("="*70)
    print("SHACL Validation of PolicyCheckers")
    print("="*70)
    
    # Load SHACL shapes
    shapes_path = Path("policy_checker_shape.ttl")
    if not shapes_path.exists():
        print("\n⚠️ SHACL shape file not found")
    else:
        shapes_graph = Graph()
        shapes_graph.parse(str(shapes_path), format='turtle')
        print(f"\n✓ Loaded SHACL shapes: {len(shapes_graph)} triples")
        
        # Validate the result from previous cell
        print("\nValidating PolicyCheckers...")
        conforms, report_graph, report_text = validate(
            result,
            shacl_graph=shapes_graph,
            inference='rdfs',
            abort_on_first=False,
        )
        
        if conforms:
            print("\n✅ All PolicyCheckers are valid!")
            print(f"   - {len(list(result.subjects(RDF.type, tbox.PolicyChecker)))} PolicyCheckers checked")
            print(f"   - {len(list(result.subjects(RDF.type, tbox.Operation)))} Operations validated")
            print(f"   - {len(list(result.subjects(RDF.type, tbox.Report)))} Reports validated")
        else:
            print("\n❌ Validation failed!")
            print("\nViolations:")
            print(report_text)
            
except ImportError:
    print("⚠️ pyshacl not installed - run: pip install pyshacl")
except Exception as e:
    print(f"Error during validation: {e}")

Usage: python planner_n3.py <sdm_file> <data_product_name>


SystemExit: 1

/home/acraf/psr/Fdatavalidation-1/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
